In [1]:
#Load libraries

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers,Model
from sklearn.preprocessing import StandardScaler

I0000 00:00:1783014011.145031   16760 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
#Load the cleaned data

data = pd.read_csv('bad_nauheim.csv')
data.columns

Index(['Unnamed: 0', 'visitors', 'name', 'avgDuration', 'timestamp', 'SID',
       'weekday', 'hour_sin', 'hour_cos'],
      dtype='str')

In [3]:
#Configurations

features = ['visitors','avgDuration','weekday','hour_sin','hour_cos']
n_features = len(features)
window_size = 24

In [4]:
data = data.sort_values(['SID','timestamp'])
data.head()

,Unnamed: 0,visitors,name,avgDuration,timestamp,SID,weekday,hour_sin,hour_cos
28,28,1,Kurpark - Eingang Kurhaus,17.55,2025-06-30 01:00:00+00:00,26213d2d-2954-4312-b42a-a6b323764086,0,0.258819,0.965926
55,55,3,Kurpark - Eingang Kurhaus,4.10,2025-06-30 02:00:00+00:00,26213d2d-2954-4312-b42a-a6b323764086,0,0.500000,0.866025
129,129,3,Kurpark - Eingang Kurhaus,0.75,2025-06-30 03:00:00+00:00,26213d2d-2954-4312-b42a-a6b323764086,0,0.707107,0.707107
194,194,17,Kurpark - Eingang Kurhaus,6.09,2025-06-30 04:00:00+00:00,26213d2d-2954-4312-b42a-a6b323764086,0,0.866025,0.500000
197,197,17,Kurpark - Eingang Kurhaus,6.44,2025-06-30 04:00:00+00:00,26213d2d-2954-4312-b42a-a6b323764086,0,0.866025,0.500000


In [5]:
def create_windows(data, window_size = window_size):
    X = []

    for i in range(len(data)-window_size+1):
        window = data[i:i+window_size]
        X.append(window)

    return np.array(X)

In [6]:
# Make a windows for all available sensors

train_data = []

for sensor_id in data['SID'].unique():
    sensor_data = data[data['SID'] == sensor_id]
    values = sensor_data[features].values

    windows = create_windows(values,window_size=window_size)
    train_data.append(windows)

train_data = np.vstack(train_data)

print(train_data.shape)

(16888, 24, 5)


In [7]:
#Standardarizing the data

scaler = StandardScaler()

train_data_reshaped = train_data.reshape(-1,2)

In [8]:
train_data_scaled = scaler.fit_transform(train_data_reshaped)

#Convert our data into normal shape

train_data = train_data_scaled.reshape(-1,24,n_features)
print(train_data.shape)

(16888, 24, 5)


In [9]:
#Build a Autoencoder-LSTM Model

inputs = layers.Input(shape=(window_size,n_features))
x = layers.LSTM(64,return_sequences=True)(inputs)
x = layers.LSTM(32,return_sequences = False)(x)

#Then it is a bottleneck or a latent which saves the compressed knowledge
latent = layers.Dense(16,activation='relu')(x)

x = layers.RepeatVector(window_size)(latent)
x = layers.LSTM(32,return_sequences = True)(x)
x = layers.LSTM(64,return_sequences=True)(x)

outputs = layers.TimeDistributed(layers.Dense(n_features))(x)

model = Model(inputs,outputs)

I0000 00:00:1783014013.953108   16760 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4620 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1660 Ti with Max-Q Design, pci bus id: 0000:01:00.0, compute capability: 7.5


In [10]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 24, 5)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 24, 64)         │        17,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 24, 16)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 24, 32)         │         6,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 24, 64)         │        24,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 24, 5)          │           325 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 62,293 (243.33 KB)

 Trainable params: 62,293 (243.33 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
model.compile(optimizer = 'adam',loss= 'mse')

In [19]:
model.fit(train_data,train_data,
          epochs = 25,
          batch_size=64,
         validation_split=0.2)

Epoch 1/25
212/212 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - loss: 0.0423 - val_loss: 0.0417
Epoch 2/25
212/212 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - loss: 0.0420 - val_loss: 0.0309
Epoch 3/25
212/212 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - loss: 0.0389 - val_loss: 0.0363
Epoch 4/25
212/212 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - loss: 0.0396 - val_loss: 0.0323
Epoch 5/25
212/212 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - loss: 0.0392 - val_loss: 0.0305
Epoch 6/25
212/212 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - loss: 0.0395 - val_loss: 0.0294
Epoch 7/25
212/212 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - loss: 0.0372 - val_loss: 0.0290
Epoch 8/25
212/212 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - loss: 0.0360 - val_loss: 0.0292
Epoch 9/25
212/212 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - loss: 0.0354 - val_loss: 0.0290
Epoch 10/25
212/212 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - loss: 0.0361 - val_loss: 0.0291
Epoch 11/25
212/212 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - loss: 0.0354 - val_loss: 0.0400
Epoch 12/25
212/212 ━━━━━━━━━━━━━━━━━━━━ 

In [14]:
pred = model.predict(train_data)

528/528 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step


In [17]:
errors = np.mean(np.square(train_data-pred),axis = (1,2))

In [18]:
errors

array([0.01731479, 0.01793218, 0.01882565, ..., 0.01314715, 0.01288462,
       0.01361025], shape=(16888,))